In [1]:
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

In [8]:
bicicletas = ['razor','zoomer']
requisitos = ['polimero','tempo']
prod_total = 700

val_requisitos ={
    requisitos[0]:408.2 ,
    requisitos[1]:2400 ,
}

informacoes ={
    bicicletas[0]:{
        requisitos[0]:0.9,
        requisitos[1]:3,
        'lucro':70,
    },
    bicicletas[1]:{
        requisitos[0]:0.45,
        requisitos[1]:4,
        'lucro':40,
    }
}

In [24]:
model = pyo.ConcreteModel()

model.bicicletas = pyo.Set(initialize=bicicletas)
model.requisitos = pyo.Set(initialize=requisitos)
model.x = pyo.Var(model.bicicletas, bounds=(0,None), domain=pyo.NonNegativeIntegers)

def obj(model):
    return sum(
        model.x[b] * informacoes[b]['lucro'] for b in model.bicicletas
    )
model.obj = pyo.Objective(rule=obj, sense=pyo.maximize)

def restricoes(model, r):
    return sum(
        model.x[b] * informacoes[b][r]  for b in model.bicicletas
    ) <= val_requisitos[r]
model.restricoes = pyo.Constraint(model.requisitos, rule=restricoes)

def restr_quantidade_total(model):
    return sum(model.x[b] for b in model.bicicletas) <= prod_total
model.rest_total = pyo.Constraint(rule=restr_quantidade_total)

def restr_entre(model):
    return model.x[bicicletas[0]] <= 300 + model.x[bicicletas[1]]
model.restr_entre = pyo.Constraint(rule=restr_entre)

In [25]:
model.pprint()

2 Set Declarations
    bicicletas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'razor', 'zoomer'}
    requisitos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'polimero', 'tempo'}

1 Var Declarations
    x : Size=2, Index=bicicletas
        Key    : Lower : Value : Upper : Fixed : Stale : Domain
         razor :     0 :  None :  None : False :  True : NonNegativeIntegers
        zoomer :     0 :  None :  None : False :  True : NonNegativeIntegers

1 Objective Declarations
    obj : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 70*x[razor] + 40*x[zoomer]

3 Constraint Declarations
    rest_total : Size=1, Index=None, Active=True
        Key  : Lower : Body                 : Upper : Active
        None :  -Inf : x[razor] + x[zoomer] : 700.0 :   True
    restr_en

In [26]:
opt = SolverFactory('cplex')
res = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmplhrddgo2.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmpj_e_tjf8.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmpj_e_tjf8.pyomo.lp
Objective sense      : Maximize
Variables            :       2  [General Integer: 2]
Objective nonzeros   :       2
Linear constraints   :       4  [Less: 4]
  Nonzeros           :       8
  RHS nonzeros       :       4

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 40.00000       

In [27]:
for b in model.x:
    print(f"Quantidade de {b} = {pyo.value(model.x[b]):.2f}")
print(f"Lucro total R$ {model.obj():.2f}")

Quantidade de razor = 246.00
Quantidade de zoomer = 415.00
Lucro total R$ 33820.00
